# Sumativa 4 · Taller III — Calidad y análisis de ventas con Databricks

**Universidad:** Universidad Andrés Bello · Magíster en Ciencia de Datos e Inteligencia Artificial  
**Asignatura:** Gestión de Datos y Tecnologías Big Data (MCDI502) · **Grupo 7**  
**Integrantes:** Cristian Grandón, Felipe Palma y Jorge Gutiérrez  
**Entregable:** notebook de Databricks, exportado como `mcdi502_s4_grupo7.ipynb`, comprimido en ZIP.

**Caso:** integrar las ventas del trimestre de **Viña del Mar, Santiago y Concepción**, limpiar datos incompletos e inconsistentes, validar precios y outliers, preparar un dataset de ventas confiable y responder tres preguntas con **Spark SQL**.

> **Entorno obligatorio:** todo el desarrollo debe ejecutarse en un notebook de **Databricks**, no solamente en Jupyter local. Las celdas no contienen resultados prefabricados: importa el notebook, sube los CSV del curso, ajusta las rutas y utiliza «Run all» para guardar evidencias reales.

## 1. Carga de tres CSV en Databricks — ID3.1 (27 pts)

1. Descarga de Canvas los archivos `ventas_vina_del_mar.csv`, `ventas_santiago.csv` y `ventas_concepcion.csv`.
2. Súbelos al **DBFS** del espacio de Databricks, o a un Volume de Unity Catalog si tu cuenta ya no permite usar `FileStore`.
3. Modifica **solo** `RUTA_BASE` si están en otra ubicación.
4. Ejecuta y verifica que los tres archivos estén presentes. El uso de `spark` y `dbutils` confirma que el entorno es Databricks.

La ingesta inicial se hace en texto para no perder información al inferir incorrectamente fechas o precios con símbolos.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

try:
    spark
    dbutils
except NameError as exc:
    raise RuntimeError('Este notebook se debe ejecutar en Databricks, con spark y dbutils disponibles.') from exc

# Cambiar únicamente si los CSV fueron cargados a otro directorio DBFS o /Volumes/...
RUTA_BASE = 'dbfs:/FileStore/tables'
ARCHIVOS = {
    'Vina del Mar': 'ventas_vina_del_mar.csv',
    'Santiago': 'ventas_santiago.csv',
    'Concepcion': 'ventas_concepcion.csv',
}
RUTAS = {nombre: f'{RUTA_BASE.rstrip("/")}/{archivo}' for nombre, archivo in ARCHIVOS.items()}
try:
    nombres_disponibles = {info.name.rstrip('/') for info in dbutils.fs.ls(RUTA_BASE)}
except Exception as exc:
    raise FileNotFoundError(f'No se puede acceder a {RUTA_BASE}. Revisa DBFS/Volumes y la ruta base.') from exc
for nombre, ruta in RUTAS.items():
    if ARCHIVOS[nombre] not in nombres_disponibles:
        raise FileNotFoundError(f'Falta {ARCHIVOS[nombre]} dentro de {RUTA_BASE}. Sube el CSV antes de ejecutar.')
    print('Archivo localizado:', nombre, ruta)
print('Spark en Databricks:', spark.version)

### 1.1 Cargar e inspeccionar cada DataFrame

La rúbrica exige mostrar por separado el esquema (`printSchema()`) y las **primeras cinco filas** (`show(5)`) de cada sucursal. Además se presentan los conteos individuales.

In [ ]:
OPCIONES_CSV = {'header': 'true', 'inferSchema': 'false', 'mode': 'FAILFAST'}
df_vina = spark.read.options(**OPCIONES_CSV).csv(RUTAS['Vina del Mar'])
df_santiago = spark.read.options(**OPCIONES_CSV).csv(RUTAS['Santiago'])
df_concepcion = spark.read.options(**OPCIONES_CSV).csv(RUTAS['Concepcion'])

for nombre, df in [('Viña del Mar', df_vina), ('Santiago', df_santiago), ('Concepción', df_concepcion)]:
    print(f'\n--- {nombre} ---')
    print('Cantidad de filas:', df.count())
    df.printSchema()
    df.show(5, truncate=False)

### 1.2 Unificar y comprobar integridad

`unionByName()` concatena los tres DataFrames conservando el significado de las columnas por nombre. Antes de limpiar, la cantidad de filas de la unión debe ser **exactamente** la suma de las tres fuentes.

In [ ]:
filas_fuente = {'Viña del Mar': df_vina.count(), 'Santiago': df_santiago.count(), 'Concepción': df_concepcion.count()}
df_ventas_unificado = df_vina.unionByName(df_santiago).unionByName(df_concepcion)
filas_antes = df_ventas_unificado.count()
filas_esperadas = sum(filas_fuente.values())
print('Filas originales:', filas_fuente)
print('Suma:', filas_esperadas, '| DataFrame unificado:', filas_antes)
assert filas_antes == filas_esperadas, 'La unión perdió o duplicó filas.'
print('Integridad de la unión: OK')
df_ventas_unificado.show(5, truncate=False)

## 2. Línea base de calidad (antes de limpiar) — ID4.2

Se consideran faltantes tanto `NULL` como cadenas vacías. Además, se cuentan registros completamente duplicados y cantidades inválidas o no positivas **antes de modificar los datos**, para disponer de un punto de comparación real.

In [ ]:
COLUMNAS_ORIGINALES = df_ventas_unificado.columns

def contar_faltantes(df, columnas):
    expresiones = [F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast('string')) == ''), 1).otherwise(0)).alias(c) for c in columnas]
    fila = df.agg(*expresiones).first().asDict()
    return {k: int(v or 0) for k, v in fila.items()}

nulos_antes = contar_faltantes(df_ventas_unificado, COLUMNAS_ORIGINALES)
duplicados_antes = filas_antes - df_ventas_unificado.dropDuplicates().count()
print('Filas iniciales:', filas_antes)
print('Duplicados completos:', duplicados_antes)
print('Nulos/vacíos por columna antes:')
spark.createDataFrame([(c, n) for c, n in nulos_antes.items()], ['columna', 'faltantes_antes']).show(truncate=False)

## 3. Limpieza y depuración — ID4.2 (27 pts)

**Decisiones explícitas:**

- **Duplicados:** eliminar solo filas completamente repetidas mediante `.dropDuplicates()`.
- **Categoría faltante:** imputar `Sin categoría` para no perder ventas válidas, y distinguir dato desconocido de categoría real.
- **Producto:** normalizar a minúsculas y reducir espacios redundantes.
- **Fecha:** transformar los formatos presentes a `DateType` con conversiones tolerantes (`try_to_timestamp`).
- **Precio unitario:** retirar `$` y espacios, normalizar separadores de miles y decimales y convertir a `DoubleType` sin errores silenciosos.
- **Cantidad:** convertir a `IntegerType`. Registros inválidos se controlan explícitamente en la parte 4.

Se guardan temporalmente las columnas originales de fecha, precio y cantidad para justificar rechazos y facilitar auditoría.

In [ ]:
df_limpio = df_ventas_unificado.dropDuplicates()
filas_sin_duplicados = df_limpio.count()
assert filas_antes - filas_sin_duplicados == duplicados_antes

# Normalizar categorías y productos sin eliminar filas por categorías desconocidas.
df_limpio = (df_limpio
    .withColumn('categoria', F.when(F.trim(F.col('categoria')) == '', F.lit(None)).otherwise(F.trim(F.col('categoria'))))
    .fillna({'categoria': 'Sin categoría'})
    .withColumn('producto_nombre', F.lower(F.trim(F.regexp_replace(F.col('producto_nombre'), r'\s+', ' '))))
    .withColumn('_fecha_original', F.col('fecha'))
    .withColumn('_precio_original', F.col('precio_unitario'))
    .withColumn('_cantidad_original', F.col('cantidad')))

# Fecha: múltiples formatos observados en los CSV de la asignatura.
FORMATOS_FECHA = ['dd/MM/yyyy', 'dd-MM-yyyy', 'yyyy/MM/dd', 'yyyy-MM-dd']
FECHA_PARSEO = F.coalesce(*[
    F.expr(f"try_to_timestamp(trim(fecha), '{patron}')").cast('date')
    for patron in FORMATOS_FECHA
])

# Precio: $, espacios, separadores de miles (1.234 / 1,234) y coma decimal.
precio_base = F.regexp_replace(F.trim(F.col('precio_unitario')), r'[$\s]', '')
precio_limpio = (
    F.when(precio_base.rlike(r'^\d{1,3}(\.\d{3})+(,\d+)?$'),
           F.regexp_replace(F.regexp_replace(precio_base, r'\.', ''), ',', '.'))
    .when(precio_base.rlike(r'^\d{1,3}(,\d{3})+(\.\d+)?$'),
          F.regexp_replace(precio_base, ',', ''))
    .when(precio_base.rlike(r'^\d+,\d+$'), F.regexp_replace(precio_base, ',', '.'))
    .otherwise(precio_base)
)

df_limpio = (df_limpio
    .withColumn('fecha', FECHA_PARSEO)
    .withColumn('_precio_texto_limpio', precio_limpio)
    .withColumn('precio_unitario', F.expr('try_cast(_precio_texto_limpio AS DOUBLE)'))
    .withColumn('cantidad', F.expr('try_cast(trim(cantidad) AS INT)'))
    .drop('_precio_texto_limpio'))

print('Eliminación de duplicados:', duplicados_antes)
print('Filas después de deduplicar:', df_limpio.count())
print('Esquema corregido:')
df_limpio.printSchema()
df_limpio.select('fecha', 'producto_nombre', 'categoria', 'precio_unitario', 'cantidad').show(5, truncate=False)

## 4. Transformación: validación de filas — ID4.1 / ID4.3 (27 pts)

La pauta pide excluir ventas con **cantidad ≤ 0**, crear `precio_total = cantidad * precio_unitario`, y `dia_semana` desde la fecha corregida. Además se evitan totales negativos y se identifican filas sin fecha o precio numérico.

Para que el resultado sea auditable, todas las filas retiradas quedan en `df_rechazadas` junto con su motivo. Se comprueba la reconciliación: filas originales menos duplicados = aceptadas + rechazadas.

In [ ]:
df_transformado = (df_limpio
    .withColumn('precio_total', F.col('cantidad') * F.col('precio_unitario'))
    .withColumn('dia_semana', F.date_format('fecha', 'EEEE'))
    .withColumn('motivo_rechazo',
        F.when(F.col('cantidad').isNull(), 'Cantidad inválida/nula')
         .when(F.col('cantidad') <= 0, 'Cantidad <= 0')
         .when(F.col('fecha').isNull(), 'Fecha inválida/nula')
         .when(F.col('precio_unitario').isNull(), 'Precio inválido/nulo')
         .when(F.col('precio_total') < 0, 'Precio total negativo')))

df_rechazadas = df_transformado.filter(F.col('motivo_rechazo').isNotNull())
df_ventas_final = (df_transformado
    .filter(F.col('motivo_rechazo').isNull())
    .drop('motivo_rechazo', '_fecha_original', '_precio_original', '_cantidad_original'))

filas_rechazadas = df_rechazadas.count()
filas_finales = df_ventas_final.count()
print('Filas sin duplicados:', filas_sin_duplicados, '| rechazadas:', filas_rechazadas, '| finales:', filas_finales)
assert filas_sin_duplicados == filas_rechazadas + filas_finales
assert df_ventas_final.filter(F.col('cantidad') <= 0).count() == 0
assert df_ventas_final.filter(F.col('precio_total') < 0).count() == 0
print('Motivos de rechazo:')
df_rechazadas.groupBy('motivo_rechazo').count().show(truncate=False)
print('Precio total y día de semana:')
df_ventas_final.select('id_transaccion', 'fecha', 'dia_semana', 'cantidad', 'precio_unitario', 'precio_total').show(10, truncate=False)

### 4.1 Análisis `distinct()` de sucursales y categorías

La pauta exige contar y **mostrar** los valores únicos tras la limpieza.

In [ ]:
print('Sucursales distintas:', df_ventas_final.select('sucursal').distinct().count())
df_ventas_final.select('sucursal').distinct().orderBy('sucursal').show(truncate=False)
print('Categorías distintas:', df_ventas_final.select('categoria').distinct().count())
df_ventas_final.select('categoria').distinct().orderBy('categoria').show(truncate=False)

## 5. Detección de valores atípicos con IQR — ID4.3

Regla solicitada: detectar precios unitarios fuera de **[Q1 − 1,5×IQR, Q3 + 1,5×IQR]**. Los registros quedan **marcados** en `es_outlier_precio` y se muestran para su revisión; no se descartan automáticamente, porque un precio extremo puede ser legítimo.

Se exige además comprobar que `precio_total` no contenga negativos después de los filtros.

In [ ]:
if filas_finales == 0:
    raise ValueError('No hay ventas válidas sobre las que calcular cuartiles.')
Q1, Q3 = df_ventas_final.approxQuantile('precio_unitario', [0.25, 0.75], 0.0)
IQR = Q3 - Q1
LIMITE_INF, LIMITE_SUP = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

df_ventas_final = df_ventas_final.withColumn(
    'es_outlier_precio',
    (F.col('precio_unitario') < F.lit(LIMITE_INF)) | (F.col('precio_unitario') > F.lit(LIMITE_SUP))
)
df_outliers = df_ventas_final.filter('es_outlier_precio')
print(f'Q1={Q1:,.2f}, Q3={Q3:,.2f}, IQR={IQR:,.2f}')
print(f'Limites: {LIMITE_INF:,.2f} a {LIMITE_SUP:,.2f}')
print('Outliers:', df_outliers.count())
df_outliers.select('id_transaccion', 'sucursal', 'producto_nombre', 'precio_unitario', 'precio_total').show(20, truncate=False)
assert df_ventas_final.filter(F.col('precio_total') < 0).count() == 0

## 6. Informe comparativo de calidad (antes y después) — ID4.3

Se incluyen los tres indicadores mínimos que exige la rúbrica: **cantidad total de filas, nulos/vacíos por columna y duplicados**, antes y después. Se agregan rechazos, outliers y controles de consistencia. Las medidas **no** se fijan manualmente: se calculan de los DataFrames.

In [ ]:
filas_despues = df_ventas_final.count()
se_duplican = filas_despues - df_ventas_final.dropDuplicates().count()
nulos_despues = contar_faltantes(df_ventas_final, df_ventas_final.columns)

resumen_calidad = spark.createDataFrame([
    ('Filas totales', str(filas_antes), str(filas_despues)),
    ('Filas duplicadas', str(duplicados_antes), str(se_duplican)),
    ('Categorias nulas/vacias', str(nulos_antes.get('categoria', 0)), str(nulos_despues.get('categoria', 0))),
    ('Cantidad <= 0', 'No evaluado en origen', str(df_ventas_final.filter(F.col('cantidad') <= 0).count())),
    ('Precio total negativo', 'No calculado en origen', str(df_ventas_final.filter(F.col('precio_total') < 0).count())),
    ('Filas rechazadas por reglas', 'No aplica', str(filas_rechazadas)),
    ('Outliers unitarios (IQR)', 'No aplica', str(df_outliers.count())),
], ['indicador', 'antes', 'despues'])
print('INFORME GENERAL DE CALIDAD:')
resumen_calidad.show(truncate=False)

comparacion_nulos = spark.createDataFrame([
    (c, str(nulos_antes[c]) if c in nulos_antes else 'No aplica', str(nulos_despues[c]))
    for c in df_ventas_final.columns
], ['columna', 'faltantes_antes', 'faltantes_despues'])
print('NULOS/VACIOS POR COLUMNA:')
comparacion_nulos.show(truncate=False)

print('RECONCILIACION:', filas_antes, '-', duplicados_antes, '-', filas_rechazadas, '=', filas_despues)
assert filas_antes - duplicados_antes - filas_rechazadas == filas_despues
assert se_duplican == 0
print('VALIDACIONES DE CALIDAD: OK')

## 7. Análisis con Spark SQL — ID3.3 (15 pts)

Se crea la vista temporal con nombre obligatorio **`ventas_limpias`**. Las siguientes tres celdas ejecutan consultas `spark.sql()` (no sustitutos de DataFrame API), y presentan sus resultados con `.show()`.

In [ ]:
df_ventas_final.createOrReplaceTempView('ventas_limpias')
print('Vista temporal creada: ventas_limpias')

### Consulta 1: las 10 transacciones con mayor precio total

Se muestran `id_transaccion`, `fecha`, `sucursal`, `producto_nombre` y `precio_total`.

In [ ]:
consulta_1 = spark.sql('''
SELECT id_transaccion, fecha, sucursal, producto_nombre, precio_total
FROM ventas_limpias
ORDER BY precio_total DESC, id_transaccion ASC
LIMIT 10
''')
consulta_1.show(10, truncate=False)

### Consulta 2: total de ventas y número de transacciones por sucursal

In [ ]:
consulta_2 = spark.sql('''
SELECT sucursal, SUM(precio_total) AS total_ventas,
       COUNT(*) AS cantidad_transacciones
FROM ventas_limpias
GROUP BY sucursal
ORDER BY total_ventas DESC
''')
consulta_2.show(truncate=False)

### Consulta 3: promedio, mínimo y máximo de precio unitario por categoría

In [ ]:
consulta_3 = spark.sql('''
SELECT categoria,
       ROUND(AVG(precio_unitario), 2) AS precio_promedio,
       MIN(precio_unitario) AS precio_minimo,
       MAX(precio_unitario) AS precio_maximo
FROM ventas_limpias
GROUP BY categoria
ORDER BY categoria
''')
consulta_3.show(truncate=False)

## 8. Interpretación y justificación de resultados

1. **Integridad:** la suma de los tres CSV debe coincidir con `df_ventas_unificado` antes de limpiar. La diferencia entre filas iniciales y finales debe explicarse por duplicados y filas rechazadas.
2. **Categorías faltantes:** imputar `Sin categoría` conserva ventas, pero esa etiqueta **no** representa una categoría de producto auténtica.
3. **Valores atípicos:** un precio superior al umbral IQR debe revisarse, no eliminarse automáticamente. Los totales de ventas de la Consulta 2 aún incluyen outliers **marcados**.
4. **SQL:** consulta 1 identifica transacciones grandes, consulta 2 permite comparar sucursales y consulta 3 resume precios de productos por categoría.

Se generan cifras desde los DataFrames ya procesados para evitar interpretar resultados antiguos si los CSV cambian.

In [ ]:
ventas_totales = df_ventas_final.agg(F.sum('precio_total')).first()[0] or 0
ventas_atipicas = df_outliers.agg(F.sum('precio_total')).first()[0] or 0
porcentaje_atipico = 100 * ventas_atipicas / ventas_totales if ventas_totales else 0
print(f'Filas: origen={filas_antes}, sin duplicados={filas_sin_duplicados}, finales={filas_despues}')
print('Outliers marcados:', df_outliers.count())
print('Monto total vendido:', ventas_totales)
print('Monto asociado a outliers:', ventas_atipicas)
print('Porcentaje del total asociado a outliers: {:.2f}%'.format(porcentaje_atipico))
print('Lectura de resultados: los importes atípicos pueden distorsionar los totales; requieren revisión comercial.')

## 9. Checklist de entrega (108 puntos)

| Criterio de la pauta | Evidencia dentro de este notebook | Máximo |
|---|---|---:|
| Carga e integración (ID3.1) | 3 CSV, 3 `.printSchema()`, 3 `.show(5)`, unión y conteos | 27 |
| Limpieza y depuración (ID4.2) | Categorías, duplicados, nombres, fechas, precio | 27 |
| Transformación y validación (ID4.1, ID4.3) | cantidad positiva, 2 columnas, `distinct`, IQR, informe calidad | 27 |
| Spark SQL (ID3.3) | Vista `ventas_limpias` y 3 consultas visibles | 15 |
| Documentación y comentarios | Markdown con decisiones y explicaciones | 12 |
| **Total** | | **108** |

**Instrucciones finales:** ejecutar **todas** las celdas dentro de Databricks, comprobar que no existan errores, guardar salidas, **exportar** el notebook con nombre `mcdi502_s4_grupo7.ipynb`, comprimirlo en un **único ZIP** y subirlo a Canvas. La Sumativa 4 **no exige video** ni informe Word adicional.